In [1]:
!pip install pytest behave coverage


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [behave]


In [2]:
%%writefile test_inventario.py
import pytest
from inventario import InventoryService, InsufficientStockException

def test_stock_negativo_lanza_excepcion():
    s = InventoryService({"P1": 5})
    with pytest.raises(InsufficientStockException):
        s.update_stock("P1", -10)

def test_retiro_valido_actualiza_stock():
    s = InventoryService({"P1": 5})
    s.update_stock("P1", -3)
    assert s.stock["P1"] == 2


Writing test_inventario.py


In [3]:
!pytest -v


============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.15.1, langsmith-0.14.15, platformdirs-4.14.0, typeguard-4.6.0
collected 0 items / 1 error                                                    

==================================== ERRORS ====================================
_____________________ ERROR collecting test_inventario.py ______________________
ImportError while importing test module '/content/test_inventario.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
test_inventario.py:2: in <module>
    from inventario import InventoryService, InsufficientStockException
E   ModuleNotFoundError: No

In [4]:
%%writefile inventario.py
class InsufficientStockException(Exception):
    pass

class InventoryService:
    def __init__(self, stock):
        self.stock = stock

    def update_stock(self, product_id, quantity):
        nuevo = self.stock[product_id] + quantity
        if nuevo < 0:
            raise InsufficientStockException("Stock insuficiente")
        self.stock[product_id] = nuevo

Writing inventario.py


In [5]:
!pytest -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.15.1, langsmith-0.14.15, platformdirs-4.14.0, typeguard-4.6.0
collected 2 items                                                              

test_inventario.py::test_stock_negativo_lanza_excepcion PASSED           [ 50%]
test_inventario.py::test_retiro_valido_actualiza_stock PASSED            [100%]

============================== 2 passed in 0.02s ===============================


In [6]:
%%writefile test_inventario.py
import pytest
from inventario import InventoryService, InsufficientStockException

def test_stock_negativo_lanza_excepcion():
    s = InventoryService({"P1": 5})
    with pytest.raises(InsufficientStockException):
        s.update_stock("P1", -10)

def test_retiro_valido_actualiza_stock():
    s = InventoryService({"P1": 5})
    s.update_stock("P1", -3)
    assert s.stock["P1"] == 2

def test_producto_inexistente_lanza_excepcion():
    s = InventoryService({"P1": 5})
    with pytest.raises(ValueError):
        s.update_stock("P99", 1)


Overwriting test_inventario.py


In [7]:
!pytest -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.15.1, langsmith-0.14.15, platformdirs-4.14.0, typeguard-4.6.0
collected 3 items                                                              

test_inventario.py::test_stock_negativo_lanza_excepcion PASSED           [ 33%]
test_inventario.py::test_retiro_valido_actualiza_stock PASSED            [ 66%]
test_inventario.py::test_producto_inexistente_lanza_excepcion FAILED     [100%]

=================================== FAILURES ===================================
__________________ test_producto_inexistente_lanza_excepcion ___________________

    def test_producto_inexistente_lanza_excepcion():
        s = InventoryService({"P1": 5})
        with pytest.raises(ValueError):
>           s.update_stock("P99", 1)

test_inventario.py:17: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _

In [8]:
%%writefile inventario.py
class InsufficientStockException(Exception):
    pass

class InventoryService:
    def __init__(self, stock):
        self.stock = stock

    def _obtener_stock(self, product_id):
        if product_id not in self.stock:
            raise ValueError(f"Producto {product_id} no existe")
        return self.stock[product_id]

    def update_stock(self, product_id, quantity):
        actual = self._obtener_stock(product_id)
        nuevo = actual + quantity
        if nuevo < 0:
            raise InsufficientStockException(
                f"Stock insuficiente para {product_id}: hay {actual}, se pidió {-quantity}"
            )
        self.stock[product_id] = nuevo

Overwriting inventario.py


In [9]:
!pytest -v


============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.15.1, langsmith-0.14.15, platformdirs-4.14.0, typeguard-4.6.0
collected 3 items                                                              

test_inventario.py::test_stock_negativo_lanza_excepcion PASSED           [ 33%]
test_inventario.py::test_retiro_valido_actualiza_stock PASSED            [ 66%]
test_inventario.py::test_producto_inexistente_lanza_excepcion PASSED     [100%]

============================== 3 passed in 0.02s ===============================


In [11]:
%%writefile test_proveedor.py
import unittest
from proveedor_service import SupplierService, InvalidSupplierException

class TestSupplierService(unittest.TestCase):

    def test_crea_proveedor_con_datos_validos(self):
        service = SupplierService()
        proveedor = service.create_supplier("900123456", "ACME S.A.S.", "contacto@acme.com")
        self.assertEqual(proveedor["nit"], "900123456")

    def test_rechaza_nit_invalido(self):
        service = SupplierService()
        with self.assertRaises(InvalidSupplierException):
            service.create_supplier("123", "ACME S.A.S.", "contacto@acme.com")

    def test_rechaza_correo_invalido(self):
        service = SupplierService()
        with self.assertRaises(InvalidSupplierException):
            service.create_supplier("900123456", "ACME S.A.S.", "sin-arroba.com")

Overwriting test_proveedor.py


In [12]:
!python -m unittest test_proveedor -v

test_proveedor (unittest.loader._FailedTest.test_proveedor) ... ERROR

ERROR: test_proveedor (unittest.loader._FailedTest.test_proveedor)
----------------------------------------------------------------------
ImportError: Failed to import test module: test_proveedor
Traceback (most recent call last):
  File "/usr/lib/python3.13/unittest/loader.py", line 141, in loadTestsFromName
    module = __import__(module_name)
  File "/content/test_proveedor.py", line 2, in <module>
    from proveedor_service import SupplierService, InvalidSupplierException
ModuleNotFoundError: No module named 'proveedor_service'


----------------------------------------------------------------------
Ran 1 test in 0.001s

FAILED (errors=1)


In [13]:
%%writefile proveedor_service.py
class InvalidSupplierException(Exception):
    pass

class SupplierService:
    def create_supplier(self, nit, nombre, correo):
        if nit is None or len(nit) < 9:
            raise InvalidSupplierException("NIT inválido")
        if "@" not in correo:
            raise InvalidSupplierException("Correo inválido")
        return {"nit": nit, "nombre": nombre, "correo": correo}

Writing proveedor_service.py


In [14]:
!python -m unittest test_proveedor -v

test_crea_proveedor_con_datos_validos (test_proveedor.TestSupplierService.test_crea_proveedor_con_datos_validos) ... ok
test_rechaza_correo_invalido (test_proveedor.TestSupplierService.test_rechaza_correo_invalido) ... ok
test_rechaza_nit_invalido (test_proveedor.TestSupplierService.test_rechaza_nit_invalido) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.000s

OK


In [15]:
%%writefile test_proveedor.py
import unittest
from proveedor_service import SupplierService, InvalidSupplierException

class TestSupplierService(unittest.TestCase):

    def test_crea_proveedor_con_datos_validos(self):
        service = SupplierService()
        proveedor = service.create_supplier("900123456", "ACME S.A.S.", "contacto@acme.com")
        self.assertEqual(proveedor["nit"], "900123456")

    def test_rechaza_nit_invalido(self):
        service = SupplierService()
        with self.assertRaises(InvalidSupplierException):
            service.create_supplier("123", "ACME S.A.S.", "contacto@acme.com")

    def test_rechaza_correo_invalido(self):
        service = SupplierService()
        with self.assertRaises(InvalidSupplierException):
            service.create_supplier("900123456", "ACME S.A.S.", "sin-arroba.com")

    def test_rechaza_correos_mal_formados(self):
        service = SupplierService()
        for correo in ["usuario@", "@acme.com", "a@b"]:
            with self.subTest(correo=correo):
                with self.assertRaises(InvalidSupplierException):
                    service.create_supplier("900123456", "ACME S.A.S.", correo)

Overwriting test_proveedor.py


In [16]:
!python -m unittest test_proveedor -v

test_crea_proveedor_con_datos_validos (test_proveedor.TestSupplierService.test_crea_proveedor_con_datos_validos) ... ok
test_rechaza_correo_invalido (test_proveedor.TestSupplierService.test_rechaza_correo_invalido) ... ok
test_rechaza_correos_mal_formados (test_proveedor.TestSupplierService.test_rechaza_correos_mal_formados) ... 
  test_rechaza_correos_mal_formados (test_proveedor.TestSupplierService.test_rechaza_correos_mal_formados) (correo='usuario@') ... FAIL
  test_rechaza_correos_mal_formados (test_proveedor.TestSupplierService.test_rechaza_correos_mal_formados) (correo='@acme.com') ... FAIL
  test_rechaza_correos_mal_formados (test_proveedor.TestSupplierService.test_rechaza_correos_mal_formados) (correo='a@b') ... FAIL
test_rechaza_nit_invalido (test_proveedor.TestSupplierService.test_rechaza_nit_invalido) ... ok

FAIL: test_rechaza_correos_mal_formados (test_proveedor.TestSupplierService.test_rechaza_correos_mal_formados) (correo='usuario@')
------------------------------------

In [17]:
%%writefile proveedor_service.py
import re

class InvalidSupplierException(Exception):
    pass

class SupplierService:
    PATRON_CORREO = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

    def _validar_nit(self, nit):
        if nit is None or len(nit) < 9:
            raise InvalidSupplierException("NIT inválido")

    def _validar_correo(self, correo):
        if not correo or not self.PATRON_CORREO.match(correo):
            raise InvalidSupplierException("Correo inválido")

    def create_supplier(self, nit, nombre, correo):
        self._validar_nit(nit)
        self._validar_correo(correo)
        return {"nit": nit, "nombre": nombre, "correo": correo}

Overwriting proveedor_service.py


In [18]:
%%writefile proveedor_service.py
import re

class InvalidSupplierException(Exception):
    pass

class SupplierService:
    PATRON_CORREO = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

    def _validar_nit(self, nit):
        if nit is None or len(nit) < 9:
            raise InvalidSupplierException("NIT inválido")

    def _validar_correo(self, correo):
        if not correo or not self.PATRON_CORREO.match(correo):
            raise InvalidSupplierException("Correo inválido")

    def create_supplier(self, nit, nombre, correo):
        self._validar_nit(nit)
        self._validar_correo(correo)
        return {"nit": nit, "nombre": nombre, "correo": correo}

Overwriting proveedor_service.py


In [19]:
!python -m unittest test_proveedor -v

test_crea_proveedor_con_datos_validos (test_proveedor.TestSupplierService.test_crea_proveedor_con_datos_validos) ... ok
test_rechaza_correo_invalido (test_proveedor.TestSupplierService.test_rechaza_correo_invalido) ... ok
test_rechaza_correos_mal_formados (test_proveedor.TestSupplierService.test_rechaza_correos_mal_formados) ... ok
test_rechaza_nit_invalido (test_proveedor.TestSupplierService.test_rechaza_nit_invalido) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.001s

OK


In [22]:
!mkdir -p features/steps

In [23]:
%%writefile features/cotizaciones.feature
# language: es
Característica: Registro de cotizaciones
  Como jefe de compras
  Quiero registrar cotizaciones de proveedores
  Para comparar ofertas sobre una solicitud activa

  Escenario: Registrar una cotización válida
    Dado un proveedor existente con NIT "900123456"
    Y una solicitud de cotización "SC-001" en estado "ABIERTA"
    Cuando el proveedor registra una cotización por 2500000
    Entonces la cotización queda en estado "RECIBIDA"
    Y el monto cotizado es 2500000

Writing features/cotizaciones.feature


In [24]:
%%writefile cotizacion_service.py
class InvalidQuotationException(Exception):
    pass

class QuotationService:
    def __init__(self):
        self.proveedores = set()
        self.solicitudes = {}   # codigo -> estado
        self.cotizaciones = []

    def agregar_proveedor(self, nit):
        self.proveedores.add(nit)

    def agregar_solicitud(self, codigo, estado):
        self.solicitudes[codigo] = estado

    def registrar_cotizacion(self, nit, codigo_solicitud, monto):
        if nit not in self.proveedores:
            raise InvalidQuotationException("Proveedor no existe")
        if self.solicitudes.get(codigo_solicitud) != "ABIERTA":
            raise InvalidQuotationException("La solicitud no está abierta")
        if monto <= 0:
            raise InvalidQuotationException("Monto inválido")
        cotizacion = {
            "nit": nit,
            "solicitud": codigo_solicitud,
            "monto": monto,
            "estado": "RECIBIDA",
        }
        self.cotizaciones.append(cotizacion)
        return cotizacion

Writing cotizacion_service.py


In [25]:
%%writefile features/steps/cotizaciones_steps.py
import os
import sys
sys.path.insert(0, os.getcwd())

from behave import given, when, then
from cotizacion_service import QuotationService


@given('un proveedor existente con NIT "{nit}"')
def step_proveedor_existente(context, nit):
    context.service = QuotationService()
    context.service.agregar_proveedor(nit)
    context.nit = nit


@given('una solicitud de cotización "{codigo}" en estado "{estado}"')
def step_solicitud(context, codigo, estado):
    context.service.agregar_solicitud(codigo, estado)
    context.codigo = codigo


@when('el proveedor registra una cotización por {monto:d}')
def step_registrar(context, monto):
    context.cotizacion = context.service.registrar_cotizacion(
        context.nit, context.codigo, monto
    )


@then('la cotización queda en estado "{estado}"')
def step_estado(context, estado):
    assert context.cotizacion["estado"] == estado


@then('el monto cotizado es {monto:d}')
def step_monto(context, monto):
    assert context.cotizacion["monto"] == monto

Writing features/steps/cotizaciones_steps.py


In [26]:
!behave

USING RUNNER: behave.runner:Runner
Característica: Registro de cotizaciones # features/cotizaciones.feature:2
  Como jefe de compras
  Quiero registrar cotizaciones de proveedores
  Para comparar ofertas sobre una solicitud activa
  Escenario: Registrar una cotización válida                   # features/cotizaciones.feature:7
    Dado un proveedor existente con NIT "900123456"            # features/steps/cotizaciones_steps.py:9
    Dado un proveedor existente con NIT "900123456"            # features/steps/cotizaciones_steps.py:9 0.000s
    Y una solicitud de cotización "SC-001" en estado "ABIERTA" # features/steps/cotizaciones_steps.py:16
    Y una solicitud de cotización "SC-001" en estado "ABIERTA" # features/steps/cotizaciones_steps.py:16 0.000s
    Cuando el proveedor registra una cotización por 2500000    # features/steps/cotizaciones_steps.py:22
    Cuando el proveedor registra una cotización por 2500000    # features/steps/cotizaciones_steps.py:22 0.000s
    Entonces la cotizac

In [27]:
!coverage run -m pytest test_proveedor.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: anyio-4.15.1, langsmith-0.14.15, platformdirs-4.14.0, typeguard-4.6.0
collected 4 items                                                              

test_proveedor.py ....                                                   [100%]

============================== 4 passed in 0.04s ===============================


In [28]:
!coverage report -m

Name                   Stmts   Miss  Cover   Missing
----------------------------------------------------
proveedor_service.py      15      0   100%
test_proveedor.py         21      0   100%
----------------------------------------------------
TOTAL                     36      0   100%


In [29]:
!coverage run -m behave

USING RUNNER: behave.runner:Runner
Característica: Registro de cotizaciones # features/cotizaciones.feature:2
  Como jefe de compras
  Quiero registrar cotizaciones de proveedores
  Para comparar ofertas sobre una solicitud activa
  Escenario: Registrar una cotización válida                   # features/cotizaciones.feature:7
    Dado un proveedor existente con NIT "900123456"            # features/steps/cotizaciones_steps.py:9
    Dado un proveedor existente con NIT "900123456"            # features/steps/cotizaciones_steps.py:9 0.000s
    Y una solicitud de cotización "SC-001" en estado "ABIERTA" # features/steps/cotizaciones_steps.py:16
    Y una solicitud de cotización "SC-001" en estado "ABIERTA" # features/steps/cotizaciones_steps.py:16 0.000s
    Cuando el proveedor registra una cotización por 2500000    # features/steps/cotizaciones_steps.py:22
    Cuando el proveedor registra una cotización por 2500000    # features/steps/cotizaciones_steps.py:22 0.000s
    Entonces la cotizac

In [30]:
!coverage report -m --include="cotizacion_service.py"

Name                    Stmts   Miss  Cover   Missing
-----------------------------------------------------
cotizacion_service.py      21      3    86%   18, 20, 22
-----------------------------------------------------
TOTAL                      21      3    86%


In [31]:
%%writefile test_cotizacion.py
import pytest
from cotizacion_service import QuotationService, InvalidQuotationException


@pytest.fixture
def service():
    s = QuotationService()
    s.agregar_proveedor("900123456")
    s.agregar_solicitud("SC-001", "ABIERTA")
    s.agregar_solicitud("SC-002", "CERRADA")
    return s


def test_rechaza_proveedor_inexistente(service):
    with pytest.raises(InvalidQuotationException):
        service.registrar_cotizacion("999999999", "SC-001", 1000)


def test_rechaza_solicitud_cerrada(service):
    with pytest.raises(InvalidQuotationException):
        service.registrar_cotizacion("900123456", "SC-002", 1000)


@pytest.mark.parametrize("monto", [0, -1000])
def test_rechaza_monto_invalido(service, monto):
    with pytest.raises(InvalidQuotationException):
        service.registrar_cotizacion("900123456", "SC-001", monto)

Writing test_cotizacion.py


In [32]:
!coverage run -m pytest test_cotizacion.py


============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: anyio-4.15.1, langsmith-0.14.15, platformdirs-4.14.0, typeguard-4.6.0
collected 4 items                                                              

test_cotizacion.py ....                                                  [100%]

============================== 4 passed in 0.04s ===============================


In [33]:
!coverage run -m pytest test_cotizacion.py -q
!coverage report -m --include="cotizacion_service.py"


....                                                                     [100%]
4 passed in 0.03s
Name                    Stmts   Miss  Cover   Missing
-----------------------------------------------------
cotizacion_service.py      21      3    86%   23-30
-----------------------------------------------------
TOTAL                      21      3    86%


In [34]:
!coverage erase
!coverage run -m behave -f progress
!coverage run -a -m pytest test_cotizacion.py -q
!coverage report -m --include="cotizacion_service.py"

USING RUNNER: behave.runner:Runner
features/cotizaciones.feature  .

1 feature passed, 0 failed, 0 skipped
1 scenario passed, 0 failed, 0 skipped
5 steps passed, 0 failed, 0 skipped
Took 0min 0.001s
....                                                                     [100%]
4 passed in 0.03s
Name                    Stmts   Miss  Cover   Missing
-----------------------------------------------------
cotizacion_service.py      21      0   100%
-----------------------------------------------------
TOTAL                      21      0   100%


Taller 1 - CRUD de Proveedores

In [35]:
%%writefile test_eliminar_proveedor.py
import unittest
from unittest.mock import MagicMock
from proveedor_service import SupplierService, SupplierHasActiveOrdersException


class TestEliminarProveedor(unittest.TestCase):

    def test_no_deberia_eliminar_con_ordenes_activas(self):
        repo_ordenes = MagicMock()
        repo_ordenes.has_active_orders.return_value = True
        service = SupplierService(repo_ordenes=repo_ordenes)
        with self.assertRaises(SupplierHasActiveOrdersException):
            service.delete_supplier("900123456")

    def test_deberia_desactivar_si_no_tiene_ordenes_activas(self):
        repo_ordenes = MagicMock()
        repo_ordenes.has_active_orders.return_value = False
        service = SupplierService(repo_ordenes=repo_ordenes)
        service.create_supplier("900123456", "ACME S.A.S.", "contacto@acme.com")
        proveedor = service.delete_supplier("900123456")
        self.assertEqual(proveedor["estado"], "INACTIVO")

Writing test_eliminar_proveedor.py


In [36]:
!python -m unittest test_eliminar_proveedor -v


test_eliminar_proveedor (unittest.loader._FailedTest.test_eliminar_proveedor) ... ERROR

ERROR: test_eliminar_proveedor (unittest.loader._FailedTest.test_eliminar_proveedor)
----------------------------------------------------------------------
ImportError: Failed to import test module: test_eliminar_proveedor
Traceback (most recent call last):
  File "/usr/lib/python3.13/unittest/loader.py", line 141, in loadTestsFromName
    module = __import__(module_name)
  File "/content/test_eliminar_proveedor.py", line 3, in <module>
    from proveedor_service import SupplierService, SupplierHasActiveOrdersException
ImportError: cannot import name 'SupplierHasActiveOrdersException' from 'proveedor_service' (/content/proveedor_service.py)


----------------------------------------------------------------------
Ran 1 test in 0.001s

FAILED (errors=1)


In [37]:
%%writefile proveedor_service.py
import re

class InvalidSupplierException(Exception):
    pass

class SupplierHasActiveOrdersException(Exception):
    pass

class SupplierService:
    PATRON_CORREO = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

    def __init__(self, repo_ordenes=None):
        self.repo_ordenes = repo_ordenes
        self.proveedores = {}

    def _validar_nit(self, nit):
        if nit is None or len(nit) < 9:
            raise InvalidSupplierException("NIT inválido")

    def _validar_correo(self, correo):
        if not correo or not self.PATRON_CORREO.match(correo):
            raise InvalidSupplierException("Correo inválido")

    def create_supplier(self, nit, nombre, correo):
        self._validar_nit(nit)
        self._validar_correo(correo)
        proveedor = {"nit": nit, "nombre": nombre, "correo": correo, "estado": "ACTIVO"}
        self.proveedores[nit] = proveedor
        return proveedor

    def delete_supplier(self, nit):
        if self.repo_ordenes.has_active_orders(nit):
            raise SupplierHasActiveOrdersException(
                "No se puede eliminar un proveedor con órdenes activas"
            )
        proveedor = self.proveedores[nit]
        proveedor["estado"] = "INACTIVO"
        return proveedor

Overwriting proveedor_service.py


In [38]:
!python -m unittest test_eliminar_proveedor test_proveedor -v

test_deberia_desactivar_si_no_tiene_ordenes_activas (test_eliminar_proveedor.TestEliminarProveedor.test_deberia_desactivar_si_no_tiene_ordenes_activas) ... ok
test_no_deberia_eliminar_con_ordenes_activas (test_eliminar_proveedor.TestEliminarProveedor.test_no_deberia_eliminar_con_ordenes_activas) ... ok
test_crea_proveedor_con_datos_validos (test_proveedor.TestSupplierService.test_crea_proveedor_con_datos_validos) ... ok
test_rechaza_correo_invalido (test_proveedor.TestSupplierService.test_rechaza_correo_invalido) ... ok
test_rechaza_correos_mal_formados (test_proveedor.TestSupplierService.test_rechaza_correos_mal_formados) ... ok
test_rechaza_nit_invalido (test_proveedor.TestSupplierService.test_rechaza_nit_invalido) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.002s

OK


Taller 2 - Solicitud de Cotizaciones

In [39]:
%%writefile test_taller_cotizaciones.py
import pytest
from unittest.mock import MagicMock
from cotizacion_service import QuotationService, InvalidQuotationException


@pytest.fixture
def repo():
    r = MagicMock()
    r.existe_cotizacion.return_value = False
    return r


@pytest.fixture
def service(repo):
    s = QuotationService(repo=repo)
    s.agregar_proveedor("900123456")
    s.agregar_solicitud("SC-001", "ABIERTA")
    return s


@pytest.mark.parametrize("monto", [0, -1000])
def test_monto_invalido_lanza_excepcion_y_no_guarda(service, repo, monto):
    with pytest.raises(InvalidQuotationException):
        service.registrar_cotizacion("900123456", "SC-001", monto)
    repo.save.assert_not_called()


def test_monto_valido_guarda_la_cotizacion(service, repo):
    cotizacion = service.registrar_cotizacion("900123456", "SC-001", 2500000)
    assert cotizacion["estado"] == "RECIBIDA"
    repo.save.assert_called_once()


def test_rechaza_cotizacion_duplicada(service, repo):
    repo.existe_cotizacion.return_value = True
    with pytest.raises(InvalidQuotationException):
        service.registrar_cotizacion("900123456", "SC-001", 2500000)
    repo.save.assert_not_called()

Writing test_taller_cotizaciones.py


In [40]:
!pytest test_taller_cotizaciones.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.15.1, langsmith-0.14.15, platformdirs-4.14.0, typeguard-4.6.0
collected 4 items                                                              

test_taller_cotizaciones.py::test_monto_invalido_lanza_excepcion_y_no_guarda[0] ERROR [ 25%]
test_taller_cotizaciones.py::test_monto_invalido_lanza_excepcion_y_no_guarda[-1000] ERROR [ 50%]
test_taller_cotizaciones.py::test_monto_valido_guarda_la_cotizacion ERROR [ 75%]
test_taller_cotizaciones.py::test_rechaza_cotizacion_duplicada ERROR     [100%]

==================================== ERRORS ====================================
_____ ERROR at setup of test_monto_invalido_lanza_excepcion_y_no_guarda[0] _____

repo = <MagicMock id='132012346637488'>

    @pytest.fixture
    def service(repo):
>       s = QuotationService(repo=rep

In [41]:
%%writefile cotizacion_service.py
class InvalidQuotationException(Exception):
    pass

class QuotationService:
    def __init__(self, repo=None):
        self.repo = repo
        self.proveedores = set()
        self.solicitudes = {}   # codigo -> estado
        self.cotizaciones = []

    def agregar_proveedor(self, nit):
        self.proveedores.add(nit)

    def agregar_solicitud(self, codigo, estado):
        self.solicitudes[codigo] = estado

    def registrar_cotizacion(self, nit, codigo_solicitud, monto):
        if nit not in self.proveedores:
            raise InvalidQuotationException("Proveedor no existe")
        if self.solicitudes.get(codigo_solicitud) != "ABIERTA":
            raise InvalidQuotationException("La solicitud no está abierta")
        if monto <= 0:
            raise InvalidQuotationException("Monto inválido")
        if self.repo and self.repo.existe_cotizacion(nit, codigo_solicitud):
            raise InvalidQuotationException("Cotización duplicada")
        cotizacion = {
            "nit": nit,
            "solicitud": codigo_solicitud,
            "monto": monto,
            "estado": "RECIBIDA",
        }
        self.cotizaciones.append(cotizacion)
        if self.repo:
            self.repo.save(cotizacion)
        return cotizacion

Overwriting cotizacion_service.py


In [42]:
%%writefile cotizacion_service.py
class InvalidQuotationException(Exception):
    pass

class QuotationService:
    def __init__(self, repo=None):
        self.repo = repo
        self.proveedores = set()
        self.solicitudes = {}   # codigo -> estado
        self.cotizaciones = []

    def agregar_proveedor(self, nit):
        self.proveedores.add(nit)

    def agregar_solicitud(self, codigo, estado):
        self.solicitudes[codigo] = estado

    def registrar_cotizacion(self, nit, codigo_solicitud, monto):
        if nit not in self.proveedores:
            raise InvalidQuotationException("Proveedor no existe")
        if self.solicitudes.get(codigo_solicitud) != "ABIERTA":
            raise InvalidQuotationException("La solicitud no está abierta")
        if monto <= 0:
            raise InvalidQuotationException("Monto inválido")
        if self.repo and self.repo.existe_cotizacion(nit, codigo_solicitud):
            raise InvalidQuotationException("Cotización duplicada")
        cotizacion = {
            "nit": nit,
            "solicitud": codigo_solicitud,
            "monto": monto,
            "estado": "RECIBIDA",
        }
        self.cotizaciones.append(cotizacion)
        if self.repo:
            self.repo.save(cotizacion)
        return cotizacion

Overwriting cotizacion_service.py


In [43]:
!pytest test_taller_cotizaciones.py test_cotizacion.py -v
!behave -f progress

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.15.1, langsmith-0.14.15, platformdirs-4.14.0, typeguard-4.6.0
collected 8 items                                                              

test_taller_cotizaciones.py::test_monto_invalido_lanza_excepcion_y_no_guarda[0] PASSED [ 12%]
test_taller_cotizaciones.py::test_monto_invalido_lanza_excepcion_y_no_guarda[-1000] PASSED [ 25%]
test_taller_cotizaciones.py::test_monto_valido_guarda_la_cotizacion PASSED [ 37%]
test_taller_cotizaciones.py::test_rechaza_cotizacion_duplicada PASSED    [ 50%]
test_cotizacion.py::test_rechaza_proveedor_inexistente PASSED            [ 62%]
test_cotizacion.py::test_rechaza_solicitud_cerrada PASSED                [ 75%]
test_cotizacion.py::test_rechaza_monto_invalido[0] PASSED                [ 87%]
test_cotizacion.py::test_rechaza_monto_inv

Taller 3 - Aprobación de Órdenes de Compra

In [45]:
%%writefile features/ordenes.feature
# language: es
Característica: Aprobación de órdenes de compra
  Como jefe de compras
  Quiero aprobar órdenes de compra dentro del presupuesto del proveedor
  Para controlar el gasto

  Escenario: Aprobar orden dentro del presupuesto
    Dado el proveedor "ACME" con presupuesto disponible de 10000000
    Y una orden "OC-100" por 5000000 en estado "PENDIENTE"
    Cuando el jefe de compras aprueba la orden
    Entonces la orden cambia a "APROBADA"
    Y el presupuesto del proveedor se reduce a 5000000

Writing features/ordenes.feature


In [46]:
%%writefile orden_service.py
class OrdenInvalidaException(Exception):
    pass


class OrdenService:
    def __init__(self):
        self.presupuestos = {}   # proveedor -> presupuesto disponible
        self.ordenes = {}        # codigo -> orden

    def agregar_proveedor(self, nombre, presupuesto):
        self.presupuestos[nombre] = presupuesto

    def agregar_orden(self, codigo, proveedor, monto, estado):
        self.ordenes[codigo] = {
            "codigo": codigo,
            "proveedor": proveedor,
            "monto": monto,
            "estado": estado,
        }

    def aprobar_orden(self, codigo):
        orden = self.ordenes[codigo]
        if orden["estado"] == "APROBADA":
            raise OrdenInvalidaException("La orden ya está aprobada")
        proveedor = orden["proveedor"]
        if orden["monto"] > self.presupuestos[proveedor]:
            raise OrdenInvalidaException("Excede el presupuesto del proveedor")
        self.presupuestos[proveedor] -= orden["monto"]
        orden["estado"] = "APROBADA"
        return orden

    def presupuesto_de(self, proveedor):
        return self.presupuestos[proveedor]

Writing orden_service.py


In [47]:
%%writefile features/steps/ordenes_steps.py
import os
import sys
sys.path.insert(0, os.getcwd())

from behave import given, when, then
from orden_service import OrdenService


@given('el proveedor "{nombre}" con presupuesto disponible de {presupuesto:d}')
def step_proveedor_presupuesto(context, nombre, presupuesto):
    context.service = OrdenService()
    context.service.agregar_proveedor(nombre, presupuesto)
    context.proveedor = nombre


@given('una orden "{codigo}" por {monto:d} en estado "{estado}"')
def step_orden(context, codigo, monto, estado):
    context.service.agregar_orden(codigo, context.proveedor, monto, estado)
    context.codigo_orden = codigo


@when('el jefe de compras aprueba la orden')
def step_aprobar(context):
    context.orden = context.service.aprobar_orden(context.codigo_orden)


@then('la orden cambia a "{estado}"')
def step_estado_orden(context, estado):
    assert context.orden["estado"] == estado


@then('el presupuesto del proveedor se reduce a {esperado:d}')
def step_presupuesto_reducido(context, esperado):
    actual = context.service.presupuesto_de(context.proveedor)
    assert actual == esperado, f"Esperado {esperado}, pero hay {actual}"

Writing features/steps/ordenes_steps.py


In [48]:
!behave -f progress


USING RUNNER: behave.runner:Runner
features/cotizaciones.feature  .
features/ordenes.feature  .

2 features passed, 0 failed, 0 skipped
2 scenarios passed, 0 failed, 0 skipped
10 steps passed, 0 failed, 0 skipped
Took 0min 0.001s


Taller 4 - Inventario

In [49]:
%%writefile test_taller_inventario.py
import unittest
from inventario import InventoryService, InsufficientStockException


class TestRetiroInventario(unittest.TestCase):

    def test_retiro_de_inventario(self):
        casos = [
            # codigo, stock, retiro, final, mensaje
            ("PROD-001", 10, 3, 7, "Retiro exitoso"),
            ("PROD-002", 5, 5, 0, "Retiro exitoso"),
            ("PROD-003", 2, 5, 2, "Stock insuficiente"),
        ]
        for codigo, stock, retiro, final, mensaje in casos:
            with self.subTest(codigo=codigo):
                service = InventoryService({codigo: stock})
                try:
                    resultado = service.retirar(codigo, retiro)
                except InsufficientStockException as e:
                    resultado = "Stock insuficiente"
                self.assertEqual(resultado, mensaje)
                self.assertEqual(service.stock[codigo], final)

Writing test_taller_inventario.py


In [50]:
!python -m unittest test_taller_inventario -v


test_retiro_de_inventario (test_taller_inventario.TestRetiroInventario.test_retiro_de_inventario) ... 
  test_retiro_de_inventario (test_taller_inventario.TestRetiroInventario.test_retiro_de_inventario) (codigo='PROD-001') ... ERROR
  test_retiro_de_inventario (test_taller_inventario.TestRetiroInventario.test_retiro_de_inventario) (codigo='PROD-002') ... ERROR
  test_retiro_de_inventario (test_taller_inventario.TestRetiroInventario.test_retiro_de_inventario) (codigo='PROD-003') ... ERROR

ERROR: test_retiro_de_inventario (test_taller_inventario.TestRetiroInventario.test_retiro_de_inventario) (codigo='PROD-001')
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/content/test_taller_inventario.py", line 18, in test_retiro_de_inventario
    resultado = service.retirar(codigo, retiro)
                ^^^^^^^^^^^^^^^
AttributeError: 'InventoryService' object has no attribute 'retirar'

ERROR: test_retiro_de_inventario (test_tal

In [51]:
%%writefile inventario.py
class InsufficientStockException(Exception):
    pass

class InventoryService:
    def __init__(self, stock):
        self.stock = stock

    def _obtener_stock(self, product_id):
        if product_id not in self.stock:
            raise ValueError(f"Producto {product_id} no existe")
        return self.stock[product_id]

    def update_stock(self, product_id, quantity):
        actual = self._obtener_stock(product_id)
        nuevo = actual + quantity
        if nuevo < 0:
            raise InsufficientStockException(
                f"Stock insuficiente para {product_id}: hay {actual}, se pidió {-quantity}"
            )
        self.stock[product_id] = nuevo

    def retirar(self, product_id, cantidad):
        self.update_stock(product_id, -cantidad)
        return "Retiro exitoso"

Overwriting inventario.py


In [52]:
!python -m unittest test_taller_inventario -v
!pytest test_inventario.py -q

test_retiro_de_inventario (test_taller_inventario.TestRetiroInventario.test_retiro_de_inventario) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.000s

OK
...                                                                      [100%]
3 passed in 0.02s
